# pop LoRA bridge -- one-notebook Colab run (train -> generate -> eval)

Trains the **LoRA arm** end to end: PEFT-adapts `Qwen2.5-Coder-1.5B-Instruct` on the
CodeXGLUE Java refinement pairs, generates fixes on the test split, and scores them with
`pop eval` -- so the LoRA arm lands a `results/lora_qwen_test.json` in the same schema as the
T5 and RAG arms. Every artifact is stored on **your Google Drive** so nothing is lost when a
Colab session ends.

**Before first run** (see `docs/colab-runbook.md`): create the shared Drive folder
`MyDrive/pop_cycle3/` and upload `pop_repo.zip` into it. The RAG, scaling and LoRA notebooks
share this one workspace, so the zip is uploaded here just once and every arm's
results co-locate under a single `results/` for aggregation. Build the zip locally from your
clone of https://github.com/yib7/Strats-for-Bug-Fixing with
`git archive --format=zip -o dist/pop_repo.zip HEAD` (the repo travels to Colab as a
Drive zip rather than a git clone, so no token is needed).

**Every session**: Runtime > Change runtime type > pick a GPU, then Runtime > **Run all**.
LoRA training auto-resumes from the latest checkpoint on Drive, so re-running after a
disconnect continues where it left off; the generate/eval steps are cheap to re-run.

In [ ]:
import sys

print("Python", sys.version)
assert sys.version_info >= (3, 11), "pop needs Python >= 3.11; this Colab runtime is older"
!nvidia-smi

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pop_cycle3")
for sub in ("outputs", "results", "logs"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ZIP = BASE / "pop_repo.zip"
assert ZIP.is_file(), (
    f"Upload pop_repo.zip to {ZIP.parent} first -- build it locally with\n"
    "  git archive --format=zip -o dist/pop_repo.zip HEAD\n"
    "then drag dist/pop_repo.zip into Drive/pop_cycle3/ (see docs/colab-runbook.md)."
)
print("Drive workspace ready:", BASE)

In [ ]:
%cd /content
!rm -rf /content/repo
!unzip -q /content/drive/MyDrive/pop_cycle3/pop_repo.zip -d /content/repo
%cd /content/repo
%pip install -q -e .

In [ ]:
# Point the repo's outputs/, results/, logs/ at Drive so the trained adapter, predictions,
# metrics, and progress logs all survive session resets. The repo ships a committed results/
# directory; its contents are copied onto Drive once before the swap.
import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/pop_cycle3")
REPO = Path("/content/repo")
for sub in ("outputs", "results", "logs"):
    drive_dir = BASE / sub
    repo_dir = REPO / sub
    if repo_dir.is_symlink():
        repo_dir.unlink()
    elif repo_dir.exists():
        shutil.copytree(repo_dir, drive_dir, dirs_exist_ok=True)
        shutil.rmtree(repo_dir)
    os.symlink(drive_dir, repo_dir)
print("outputs/, results/, logs/ now live on Drive:", BASE)

### Environment fix: remove the stale torchao

Colab ships an old **torchao 0.10**, which the installed `peft` hard-rejects during LoRA
injection (`Found an incompatible version of torchao ... only versions above 0.16.0 are
supported`). This arm uses **plain LoRA on the attention projections -- no torchao
quantization** -- so the clean fix is to uninstall torchao; `peft` then skips that code path
entirely. Without this cell, the `pop lora` step below crashes before training starts.


In [ ]:
# `pop lora` runs in a subprocess (`!pop lora ...`), which picks up this uninstall. Harmless
# if torchao is already absent (pip just skips it).
!pip uninstall -y -q torchao


### Optional: Weights & Biases

LoRA training logs to W&B **only if you opt in**. To enable it, add a new cell above the
training step with `import wandb; wandb.login()` and paste **your own** key when prompted --
the key is never stored in this notebook or the repo. If you skip this, training still runs:
`pop lora` auto-disables W&B reporting when the `WANDB_API_KEY` environment variable is unset
(see `pop.train.lora`). It is deliberately not a default cell so that **Run all** never stalls
waiting for input.

### Train -> generate -> eval

Three steps, each writing to Drive:
1. `pop lora` PEFT-adapts the base model and saves the adapter to `outputs/lora_qwen/best`.
2. `pop lora-generate` builds the *same* instruction+buggy prompt the model was trained on
   (shared `build_lora_prompt`, so train and inference cannot drift), generates a fix per test
   pair, and writes `outputs/lora_qwen/predictions_test.jsonl` in `{prediction, reference}`
   form.
3. `pop eval` scores those predictions into `results/lora_qwen_test.json`.

In [ ]:
# Step 1: LoRA-finetune the adapter. Auto-resumes from the latest checkpoint on Drive if a
# previous session was interrupted.
!pop lora --config configs/lora_qwen.yaml

In [ ]:
# Step 2: generate predictions on the test split (writes outputs/lora_qwen/predictions_test.jsonl).
!pop lora-generate --config configs/lora_qwen.yaml --split test

In [ ]:
# Step 3: score the predictions (writes results/lora_qwen_test.json).
!pop eval --predictions outputs/lora_qwen/predictions_test.jsonl --name lora_qwen_test

In [ ]:
import json
from pathlib import Path

result = Path("results/lora_qwen_test.json")
if result.exists():
    metrics = json.loads(result.read_text(encoding="utf-8"))["metrics"]
    print(
        f"lora_qwen_test: CodeBLEU={metrics['codebleu']:.4f} "
        f"syntax={metrics['syntax_valid_rate']:.4f} EM={metrics['em']:.4f} n={metrics['n']}"
    )
else:
    print("results/lora_qwen_test.json not found -- did the train/generate/eval cells run?")

### If the session disconnects or hits the GPU quota

Normal and expected on the free tier. Reopen this notebook and **Run all** again: `pop lora`
resumes from the latest checkpoint on Drive (`outputs/lora_qwen/checkpoint-*`), then the
generate and eval steps re-run quickly. The trained adapter, predictions, and metrics all
live under `Drive/pop_cycle3/`.